# Notebook 07 - Transformer Training

**Kelompok 08 - Kecerdasan Buatan 02**

| NPM | Nama |
|-----|------|
| 2306242395 | Calvin Wirathama Katoroy |
| 2306220532 | Syahmi Hamdani |
| 2306220323 | Christover Angelo |
| 2306221024 | Matthew Immanuel |

**Tujuan:** Melatih Transformer encoder sebagai model tambahan dengan 6 eksperimen hyperparameter (d_model, nhead, num_layers, dropout, learning_rate). History disimpan ke Drive bersama checkpoint.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import copy
import numpy as np
import torch
import yaml
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
def safe_log_experiment(record, csv_path):
    import os, pandas as pd
    from pathlib import Path
    path = Path(csv_path)
    if path.exists():
        try:
            pd.read_csv(path)
        except Exception:
            print(f'WARNING: corrupt CSV at {path}, resetting.')
            os.remove(path)
    from src.utils import log_experiment
    log_experiment(record, csv_path=csv_path)


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/tugas-akhir-ai")
    SPLITS_DIR = DRIVE_ROOT / "splits"
else:
    import yaml
    with open("../config.yaml") as f:
        _cfg = yaml.safe_load(f)
    DRIVE_ROOT = Path(_cfg["data"]["drive_root"])
    SPLITS_DIR = Path(_cfg["data"]["splits_drive"])

print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"SPLITS_DIR: {SPLITS_DIR}")

In [ ]:
with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

# Transformer uses same seq splits as LSTM/GRU
X_train = np.load(SPLITS_DIR / "X_train_seq.npy")
y_train = np.load(SPLITS_DIR / "y_train_seq.npy")
X_val   = np.load(SPLITS_DIR / "X_val_seq.npy")
y_val   = np.load(SPLITS_DIR / "y_val_seq.npy")
X_test  = np.load(SPLITS_DIR / "X_test_seq.npy")
y_test  = np.load(SPLITS_DIR / "y_test_seq.npy")

n_features = X_train.shape[2]
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"n_features={n_features}")

## Baseline run (config defaults)

In [ ]:
from src.models.transformer_model import build_transformer
from src.train import train_model
from src.utils import log_experiment

model = build_transformer(cfg, n_features)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
import os, shutil, json

_ckpt_local = '../results/checkpoints/best_transformer.pt'
_ckpt_drive = str(DRIVE_ROOT / 'best_transformer.pt')
_hist_local = '../results/checkpoints/history_transformer_01.json'
_hist_drive = str(DRIVE_ROOT / 'history_transformer_01.json')

os.makedirs('../results/checkpoints', exist_ok=True)
if not os.path.exists(_ckpt_local) and os.path.exists(_ckpt_drive):
    shutil.copy(_ckpt_drive, _ckpt_local)
    print('Restored best_transformer.pt from Drive')
    if os.path.exists(_hist_drive):
        shutil.copy(_hist_drive, _hist_local)
        print('Restored history from Drive')

if os.path.exists(_ckpt_local):
    print('Checkpoint found - skipping baseline training.')
    if os.path.exists(_hist_local):
        with open(_hist_local) as f:
            history_baseline = json.load(f)
        print('History loaded.')
    elif os.path.exists(_hist_drive):
        with open(_hist_drive) as f:
            history_baseline = json.load(f)
        print('History loaded from Drive.')
    else:
        history_baseline = {'val_f1': [0.0], 'train_loss': [], 'val_loss': []}
        print('No saved history - loss curve unavailable this session.')
else:
    history_baseline, ckpt_baseline = train_model(
        model, X_train, y_train, X_val, y_val,
        cfg=cfg, model_key='transformer',
        checkpoint_dir='../results/checkpoints',
    )
    with open(_hist_local, 'w') as f:
        json.dump({k: [float(v) for v in vals]
                   for k, vals in history_baseline.items()}, f)
    shutil.copy(_hist_local, _hist_drive)
    shutil.copy(_ckpt_local, _ckpt_drive)
    print('History and checkpoint saved to Drive.')
    safe_log_experiment({
        'exp_id': 'transformer_01_baseline',
        'model': 'Transformer',
        'd_model': cfg['transformer']['d_model'],
        'nhead': cfg['transformer']['nhead'],
        'num_layers': cfg['transformer']['num_layers'],
        'dropout': cfg['transformer']['dropout'],
        'best_val_f1': round(max(history_baseline['val_f1']), 4),
        'notes': 'baseline - config defaults',
    }, csv_path=str(DRIVE_ROOT / 'metrics_summary.csv'))


## Hyperparameter Experiments

In [ ]:
def run_transformer_experiment(exp_id, overrides, notes=''):
    import shutil
    exp_cfg = copy.deepcopy(cfg)
    exp_cfg['transformer'].update(overrides)

    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    if torch.cuda.is_available(): torch.cuda.empty_cache()
    m = build_transformer(exp_cfg, n_features)
    hist, ckpt = train_model(
        m, X_train, y_train, X_val, y_val,
        cfg=exp_cfg, model_key='transformer',
        checkpoint_dir='../results/checkpoints',
    )
    # Save per-experiment checkpoint so evaluations load the right model
    _exp_ckpt_local = f'../results/checkpoints/{exp_id}.pt'
    _exp_ckpt_drive = str(DRIVE_ROOT / f'{exp_id}.pt')
    shutil.copy(ckpt, _exp_ckpt_local)
    shutil.copy(_exp_ckpt_local, _exp_ckpt_drive)

    best_f1 = round(max(hist['val_f1']), 4)
    safe_log_experiment({
        'exp_id': exp_id,
        'model': 'TRANSFORMER',
        'hidden_size': exp_cfg['transformer'].get('hidden_size', cfg['transformer'].get('hidden_size')),
        'num_layers': exp_cfg['transformer'].get('num_layers', cfg['transformer'].get('num_layers')),
        'dropout': exp_cfg['transformer'].get('dropout', cfg['transformer'].get('dropout')),
        'seq_len': exp_cfg['transformer'].get('seq_len', cfg['transformer'].get('seq_len')),
        'lr': exp_cfg['transformer'].get('learning_rate', cfg['transformer'].get('learning_rate')),
        'batch_size': exp_cfg['transformer'].get('batch_size', cfg['transformer'].get('batch_size')),
        'best_val_f1': best_f1,
        'notes': notes,
    }, csv_path=str(DRIVE_ROOT / 'metrics_summary.csv'))
    print(f'[{exp_id}] best_val_f1={best_f1}')
    return hist, _exp_ckpt_local, best_f1


In [ ]:
# Exp 02 - larger d_model (must be divisible by nhead)
hist_02, _, f1_02 = run_transformer_experiment('transformer_02_dmodel128', {'d_model': 128}, 'd_model=128')

In [ ]:
# Exp 03 - more encoder layers
hist_03, _, f1_03 = run_transformer_experiment('transformer_03_layers4', {'num_layers': 4}, 'num_layers=4')

In [ ]:
# Exp 04 - more attention heads
hist_04, _, f1_04 = run_transformer_experiment('transformer_04_nhead8', {'nhead': 8}, 'nhead=8')

In [ ]:
# Exp 05 - higher dropout
hist_05, _, f1_05 = run_transformer_experiment('transformer_05_dropout03', {'dropout': 0.3}, 'dropout=0.3')

In [ ]:
# Exp 06 - lower learning rate
hist_06, _, f1_06 = run_transformer_experiment('transformer_06_lr0001', {'learning_rate': 1e-4}, 'lr=1e-4')

## Final Evaluation on Test Set

In [ ]:
import pandas as pd

results = pd.read_csv(str(DRIVE_ROOT / 'metrics_summary.csv'))
transformer_results = results[results['model'] == 'Transformer'].copy()
avail_cols = [c for c in transformer_results.columns if c in ['exp_id', 'd_model', 'nhead', 'num_layers', 'dropout', 'lr', 'best_val_f1', 'notes']]
print(transformer_results[avail_cols])

best_row = transformer_results.loc[transformer_results['best_val_f1'].idxmax()]
print(f'\nBest by F1: {best_row["exp_id"]}  val_F1={best_row["best_val_f1"]}')

_hist_map = {
    'transformer_01_baseline': globals().get('history_baseline'),
    'transformer_02_dmodel128': globals().get('hist_02'),
    'transformer_03_layers4': globals().get('hist_03'),
    'transformer_04_nhead8': globals().get('hist_04'),
    'transformer_05_dropout03': globals().get('hist_05'),
    'transformer_06_lr0001': globals().get('hist_06'),
}
best_history = _hist_map.get(best_row['exp_id']) or None
if not best_history or not best_history.get('train_loss'):
    for _name in ['history_baseline', 'hist_02', 'hist_03', 'hist_04', 'hist_05', 'hist_06']:
        _h = globals().get(_name)
        if _h and _h.get('train_loss'):
            best_history = _h
            print(f'Using fallback history from {_name}.')
            break
if not best_history:
    best_history = {'train_loss': [], 'val_loss': [], 'val_f1': [0.0]}
    print('No history available - loss curve will be skipped.')


In [ ]:
import copy
from src.models.transformer_model import build_transformer
from src.evaluate import evaluate_dl_model
from src.utils import plot_loss_curves

best_cfg = copy.deepcopy(cfg)
best_cfg['transformer']['d_model']    = int(best_row['d_model'])
best_cfg['transformer']['nhead']      = int(best_row['nhead'])
best_cfg['transformer']['num_layers'] = int(best_row['num_layers'])
best_cfg['transformer']['dropout']    = float(best_row['dropout'])

_ckpt_path = '../results/checkpoints/best_transformer.pt'
_state = torch.load(_ckpt_path, map_location=device)
best_model = build_transformer(best_cfg, n_features).to(device)
try:
    best_model.load_state_dict(_state)
except RuntimeError as _e:
    print(f'Config mismatch: {_e}')
    print('Falling back to default Transformer config.')
    best_cfg['transformer']['d_model']    = cfg['transformer']['d_model']
    best_cfg['transformer']['nhead']      = cfg['transformer']['nhead']
    best_cfg['transformer']['num_layers'] = cfg['transformer']['num_layers']
    best_model = build_transformer(best_cfg, n_features).to(device)
    best_model.load_state_dict(_state)
best_model.eval()

y_pred_tr, y_prob_tr = evaluate_dl_model(
    best_model, X_test, y_test,
    model_name='Transformer',
    history=best_history,
    figures_dir='../results/figures',
    csv_path=str(DRIVE_ROOT / 'metrics_summary.csv'),
)
np.save('../results/transformer_y_prob.npy', y_prob_tr)


In [ ]:
_plot_hist = best_history if best_history and best_history.get('train_loss') else None
if not _plot_hist:
    for _name in ['hist_02', 'hist_03', 'hist_04', 'hist_05', 'hist_06']:
        _h = globals().get(_name)
        if _h and _h.get('train_loss'):
            _plot_hist = _h
            print(f'Using fallback history from {_name}.')
            break

if _plot_hist:
    plot_loss_curves(
        _plot_hist['train_loss'], _plot_hist['val_loss'],
        title='Transformer Loss Curves (best)',
        save_path='../results/figures/loss_transformer.png',
    )
else:
    print('No training history available - skipping loss curve.')


In [ ]:
import shutil
shutil.copy('../results/checkpoints/best_transformer.pt', str(DRIVE_ROOT / 'best_transformer.pt'))
np.save(str(DRIVE_ROOT / 'transformer_y_prob.npy'), y_prob_tr)
print(f'Checkpoint + probs saved to Drive: {DRIVE_ROOT}')